In [ ]:
# run classical ML model on ref-alt matrix

In [1]:
from sklearn.linear_model import LassoCV, RidgeCV, LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import sys, argparse, os, numpy as np, pandas as pd
from pathlib import Path
from typing import Tuple, Dict


In [2]:
# ────────────────────────────────────────────────────────────
# 0) CONFIGURATION
# ────────────────────────────────────────────────────────────
CONFIG = {
    "FEATURE_DIR" : Path("./data/latest/feature_matrix_labels"),
    "WHO_CATALOG" : Path("./data/filtered_variants_R_only.csv"),
    "SEQ_META"    : Path("./data/catalog/protein_sequences.csv"),
    "PR_OUT_DIR"  : Path("./data/latest/results/interpretability/regression"),
}

DRUG2GENES: Dict[str, list] = {
    # ── single-gene drugs ───────────────────────────────────
    "rifampicin"   : ["rpoB"],
    "pyrazinamide" : ["pncA"],
    "capreomycin"  : ["tlyA"],
    "amikacin"     : ["eis"],
    # ── multi-gene drugs (pre-merged matrices already saved)─
    "moxifloxacin" : ["gyrA", "gyrB"],
    "streptomycin" : ["rpsL", "gid"],
    "isoniazid"    : ["katG", "inhA"],
    "ethionamide"  : ["ethA", "ethR", "inhA"],
    "ethambutol"   : ["embA", "embB", "embC"],
    "levofloxacin" : ["gyrA", "gyrB"],
}

In [3]:
# ────────────────────────────────────────────────────────────
# 1)  COMMON HELPERS
# ────────────────────────────────────────────────────────────


# --- PR Helper Functions ---

def compute_residue_scores(coef: np.ndarray) -> np.ndarray:
    return np.abs(coef)
    
def load_catalog(catalog_path, allowed_confidences):
    catalog = pd.read_csv(catalog_path)
    catalog = catalog[
        (catalog["confidence"].isin(allowed_confidences)) &
        (catalog["intersectional"] == True)
    ].copy()
    catalog["aa_pos_0idx"] = catalog["aa_pos"].astype(int) - 1
 
    return catalog

def evaluate_topk_precision_recall(drug:str, gene_name:str, scores:np.ndarray, catalog_df:pd.DataFrame, k_vals=(10,), model:str="") -> list:
        
    variants_df = catalog_df[catalog_df["gene"].str.lower() == gene_name.lower()].copy()
    if variants_df.empty:
        print(f"Skipping {gene_name}: no intersectional variants found.")
        return []

    total_actual_positives = len(np.unique(variants_df["aa_pos_0idx"]))
    print(f"Total confirmed resistance positions for {gene_name}: {total_actual_positives}")

    imp_df = pd.DataFrame({"Residue_Position": np.arange(len(scores)), "Importance": scores})
    imp_df_sorted = imp_df.sort_values("Importance", ascending=False)
    

    rows = []
    for k in k_vals:
        top_k = imp_df_sorted.head(k)
        # top_k = imp_df.nlargest(int(np.ceil(len(imp_df) * (k / 100))), "Importance")

        important_positions = set(top_k["Residue_Position"])

        true_positions = set(variants_df["aa_pos_0idx"])
        
        true_positives = len(true_positions.intersection(important_positions))
        total_predictions = len(important_positions) #k

        precision = true_positives / total_predictions if total_predictions > 0 else 0
        recall = true_positives / total_actual_positives if total_actual_positives > 0 else 0
        # f1 = 2 * prec * rec / (prec + rec + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        assert precision <= 1.0, f"Precision > 1 for {gene_name} at k={k}"

        matched_df = variants_df[variants_df["aa_pos_0idx"].isin(important_positions)]

        identified_variants = matched_df.drop_duplicates("aa_pos_0idx")["variant"].tolist()
        
        # print(f"Top residues from model {model} for top-k {k}")
        # print(top_k)

        # print("Known resistance positions from WHO (1-indexed):")
        # print(sorted(set(variants_df["aa_pos_0idx"])))


        rows.append({
            "drug": drug, "gene": gene_name, "model": model, "k": k,
            "Total_Resistance_Positions": total_actual_positives,
            "TP": true_positives,
            "precision": precision,
            "recall": recall,
            "F1": f1,
            "identified_variants": ", ".join(identified_variants) if identified_variants else "None"
        })
    return rows



def encode_labels(y):
    le = LabelEncoder()
    return le.fit_transform(y)


def load_feature_matrix_and_labels(drug_name: str):
    """
    Load a design matrix and label vector for *either* a single-gene or
    a pre-merged multi-gene drug.

    • For single-gene drugs it automatically strips the mutation-flag
      column ( column-0 and full of 0/1).
    • For multi-gene drugs it leaves the matrix untouched.
    """
    mat_f = CONFIG["FEATURE_DIR"] / f"{drug_name.upper()}_feature_matrix.npy"
    lab_f = CONFIG["FEATURE_DIR"]/ f"{drug_name.upper()}_labels.npy"

    if not (mat_f.exists() and lab_f.exists()):
        raise FileNotFoundError(
            f"Expected files not found:\n  {mat_f}\n  {lab_f}"
        )

    X = np.load(mat_f)
    y = np.load(lab_f, allow_pickle=True)

    # ── drop mutation flag IFF this is a single-gene drug ──────────
    if len(DRUG2GENES[drug_name]) == 1:
        # extra guard: be sure the first column really *is* a flag
        X = X[:, 1:]
    print(f"{drug_name}: X shape = {X.shape}")
    return X, y

def gene_slices(drug:str, n_cols:int):
    """Return {gene:(start,end)} based on reference lengths."""
    ref = pd.read_csv(CONFIG["SEQ_META"])
    lens={ g:len(ref.loc[ref["gene"]==g,"protein_sequence"].values[0])
           for g in DRUG2GENES[drug]}
    gene_slices,cursor={},0
    for g in DRUG2GENES[drug]:
        L=lens[g]; gene_slices[g]=(cursor,cursor+L); cursor+=L
    assert cursor==n_cols, f"{drug}: expected {cursor}, got {n_cols}"
    return gene_slices

In [4]:
from sklearn.metrics import roc_auc_score, mean_squared_error

def get_scores(model, X, y):
    """Helper: return predictions + safe AUC input."""
    # Try predict_proba
    if hasattr(model, "predict_proba"):
        y_pred = model.predict_proba(X)[:, 1]
    # Fallback: decision_function
    elif hasattr(model, "decision_function"):
        y_pred = model.decision_function(X)
    # Last resort: plain predict (regression or hard labels)
    else:
        y_pred = model.predict(X)
    return y_pred



In [7]:
# --- Main Training + Evaluation ---
def run_models(drug_name,k_vals=(10,)):
    print(f"\n=== Running models for {drug_name} ===")
    genes = DRUG2GENES[drug]
    X, y = load_feature_matrix_and_labels(drug_name)
    y_encoded = encode_labels(y)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
    )

    ALLOWED_CONFIDENCES = ['1) Assoc w R', '2) Assoc w R - Interim']
    who_df = load_catalog("./data/filtered_variants_output.csv", ALLOWED_CONFIDENCES)


    gene_lengths=gene_slices(drug, X.shape[1])
        


    results = {}
    pr_all_models = []

    def model_runner(name, drug_name, model, coef_extractor):
        model.fit(X_train, y_train)
    
        # --- Train predictions ---
        y_pred_train = get_scores(model, X_train, y_train)
        # --- Test predictions ---
        y_pred_test  = get_scores(model, X_test, y_test)
    
        results[name] = {
            # Test metrics
            'test_r2': model.score(X_test, y_test),
            'test_auc': roc_auc_score(y_test, y_pred_test),
            'test_mse': mean_squared_error(y_test, y_pred_test),
    
            # Train metrics
            'train_r2': model.score(X_train, y_train),
            'train_auc': roc_auc_score(y_train, y_pred_train),
            'train_mse': mean_squared_error(y_train, y_pred_train),
        }
    
        # Feature importance export (same as before)
        coefs = coef_extractor(model)
        for gene, (start, end) in gene_lengths.items():
            scores = compute_residue_scores(coefs[start:end])
            importance_df = pd.DataFrame({
                "Residue_Position": np.arange(start, end),
                "Importance": scores,
                "Model": name,
                "Gene": gene
            })
            importance_df.to_csv(
                f"data/latest/results/interpretability/regression/full_residue_scores_{gene}_{drug_name}_{name}.csv",
                index=False
            )
            pr_all_models.extend(
                evaluate_topk_precision_recall(drug_name, gene, scores, who_df, k_vals, model=name)
            )



    # ----- Lasso -----
    model_runner("lasso", drug_name, LassoCV(max_iter=10000, cv=5, random_state=42, alphas=[0.001, 0.01, 0.1, 1, 10, 100]), lambda m: m.coef_)
    model_runner("ridge", drug_name, RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10]), lambda m: m.coef_)
    model_runner("logreg", drug_name, LogisticRegressionCV(cv=3, scoring="roc_auc", max_iter=5000,
        Cs=[1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100], class_weight="balanced"), lambda m: m.coef_[0])


    # ----- Output -----
    for model_name, metrics in results.items():
        print(f"\n[{model_name.upper()}] Results for {drug_name}:")
        for metric, val in metrics.items():
            print(f"{metric}: {val:.4f}")

    # Save PR summary
    pr_df = pd.DataFrame(pr_all_models)
    pr_df.to_csv(f"data/latest/results/interpretability/regression/pr_tables/PR_{drug_name}.csv", index=False)

    return results, pr_df


In [8]:
drug_list=['rifampicin', 'pyrazinamide','capreomycin', 'amikacin','isoniazid','ethionamide','streptomycin','ethambutol','moxifloxacin','levofloxacin']

kvals=(1,5,10)
all_results = {}
all_pr = {}
for drug in drug_list:
    if drug not in DRUG2GENES:
        print(f"[skip] {drug}: not in DRUG2GENES")
        continue

    print(f"\nRunning models for {drug}")
    all_results[drug], all_pr[drug] = run_models(drug,kvals)



Running models for rifampicin

=== Running models for rifampicin ===
rifampicin: X shape = (17582, 1172)
Total confirmed resistance positions for rpoB: 26
Total confirmed resistance positions for rpoB: 26
Total confirmed resistance positions for rpoB: 26

[LASSO] Results for rifampicin:
test_r2: 0.8141
test_auc: 0.9589
test_mse: 0.0372
train_r2: 0.8304
train_auc: 0.9644
train_mse: 0.0340

[RIDGE] Results for rifampicin:
test_r2: 0.8241
test_auc: 0.9612
test_mse: 0.0352
train_r2: 0.8508
train_auc: 0.9685
train_mse: 0.0299

[LOGREG] Results for rifampicin:
test_r2: 0.9613
test_auc: 0.9613
test_mse: 0.0317
train_r2: 0.9697
train_auc: 0.9697
train_mse: 0.0289

Running models for pyrazinamide

=== Running models for pyrazinamide ===
pyrazinamide: X shape = (12842, 186)
Total confirmed resistance positions for pncA: 95
Total confirmed resistance positions for pncA: 95
Total confirmed resistance positions for pncA: 95

[LASSO] Results for pyrazinamide:
test_r2: 0.1679
test_auc: 0.6639
test_m

In [9]:
rows = []
for drug, model_dict in all_results.items():            # e.g. {'lasso':{'r2':…}}
    for model, metrics in model_dict.items():
        row = {"drug": drug, "model": model}
        row.update(metrics)                             # r2 / auc / mse
        rows.append(row)

results_df = pd.DataFrame(rows)
results_df
results_df.to_csv("data/latest/results/prediction/regression_all_results_metrics.csv", index=False)
print("saved all_results_metrics.csv")

saved all_results_metrics.csv


In [10]:
pr_df = pd.concat(all_pr.values(), ignore_index=True)
pr_df
pr_df.to_csv("data/latest/results/interpretability/regression/pr_tables/precision_recall_all_drugs.csv", index=False)
print("saved all_pr_rows.csv")

saved all_pr_rows.csv
